




# Dual-Decoder U-Net + Spatial Attention (every stage) + Cross-Attention (first & last stage)
**Joint Optic Disc & Cup Segmentation on REFUGE · input 384×384**

Author: Kartik Shivkumar Mantri · Roll No. 24UCS246 · LNMIIT / PDPM IIITDM Jabalpur

**Architecture**
- One shared U-Net encoder (DoubleConv, from scratch): 384→192→96→48→24, channels 64→1024.
- Two parallel decoders — a **disc head** (bg vs whole disc) and a **cup head** (bg vs cup).
- **Spatial attention (CBAM-style) after *every* decoder stage** in both decoders.
- **Bidirectional cross-attention** between the two decoders at the **first (coarsest, 48²)** stage and the **last (384², computed on a 48²-pooled residual)** stage.
- Fusion at inference: cup ⊆ disc → 3-class map (0 bg / 1 rim / 2 cup).

**How to run (Colab):** Runtime → Change runtime type → **GPU**. Then set the paths in the **CONFIG** cell (mount Drive or use Kaggle API), and Run all.


## 1 · Setup

In [ ]:
import os, math, random, time, glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu':
    print('WARNING: no GPU detected. In Colab: Runtime -> Change runtime type -> GPU.')

## 2 · CONFIG — set your data paths here
Point these at your REFUGE **cropped** images/masks (the same data you used in week 7).
Two easy options on Colab:

**Option A — Google Drive:** upload your REFUGE folder to Drive, mount it, and set the six paths below.

**Option B — Kaggle API:** upload your `kaggle.json`, then uncomment the Kaggle-download cell (2b) and set `DATA_ROOT` to the extracted folder.

Masks are auto-remapped to `{0:background, 1:disc, 2:cup}`. If your mask values are `0/128/255`, that is handled; if they are already `0/1/2`, that is handled too. The loader prints the unique values it finds so you can verify.

In [ ]:
# ------------------------- EDIT THESE -------------------------
IMG_SIZE   = 384
BATCH      = 4          # lower to 2 if you hit CUDA OOM at 384
EPOCHS     = 150
PATIENCE   = 25
LR         = 1e-3
WEIGHT_DECAY = 1e-4
CONTAIN_W  = 0.5        # containment-loss weight (cup must lie inside disc)
NUM_WORKERS = 2

# Folders: each split needs an images folder and a masks folder (filenames must match by stem).
TRAIN_IMG = '/content/refuge/train/images'
TRAIN_MSK = '/content/refuge/train/masks'
VAL_IMG   = '/content/refuge/val/images'
VAL_MSK   = '/content/refuge/val/masks'
TEST_IMG  = '/content/refuge/test/images'
TEST_MSK  = '/content/refuge/test/masks'

CKPT_PATH = '/content/dualdec_sa_ca_384_best.pt'
# --------------------------------------------------------------

In [ ]:
# 2a · (optional) Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# e.g. set TRAIN_IMG = '/content/drive/MyDrive/Internship/REFUGE/train/images'  (and so on)

In [ ]:
# 2b · (optional) Kaggle API download — upload kaggle.json first
# from google.colab import files; files.upload()   # choose kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle datasets download -d <your-glaucoma-datasets-slug> -p /content/data --unzip
# then set TRAIN_IMG/TRAIN_MSK/... to the extracted REFUGE cropped folders

## 3 · Dataset & augmentation

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

def remap_mask(m):
    \"\"\"Return mask in {0=bg,1=disc,2=cup}. Handles 0/128/255 and 0/1/2.\"\"\"
    u = np.unique(m)
    if set(u.tolist()) <= {0,1,2}:
        return m.astype(np.int64)
    out = np.zeros_like(m, dtype=np.int64)
    # REFUGE convention: disc (incl. cup) darker than bg; cup darkest.
    # 0/128/255 style: 0->cup? varies. We map by intensity rank: darkest area=cup, mid=disc, brightest=bg.
    # Robust rule for the common {0,128,255}: 255=bg(0), 128=disc(1), 0=cup(2).
    if set(u.tolist()) <= {0,128,255}:
        out[m==255] = 0; out[m==128] = 1; out[m==0] = 2
        return out
    # fallback: brightest = bg, darkest = cup, middle = disc
    vals = sorted(u.tolist())
    out[m==vals[-1]] = 0
    out[m==vals[0]]  = 2
    for v in vals[1:-1]:
        out[m==v] = 1
    return out

def list_pairs(img_dir, msk_dir):
    imgs = sorted(glob.glob(os.path.join(img_dir, '*')))
    pairs = []
    for ip in imgs:
        stem = os.path.splitext(os.path.basename(ip))[0]
        cands = glob.glob(os.path.join(msk_dir, stem + '.*'))
        if cands:
            pairs.append((ip, cands[0]))
    return pairs

def augment(img, mask):
    if random.random() < 0.5:
        img = img[:, ::-1].copy(); mask = mask[:, ::-1].copy()
    if random.random() < 0.5:
        img = img[::-1, :].copy(); mask = mask[::-1, :].copy()
    if random.random() < 0.5:
        ang = random.uniform(-20, 20)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w/2, h/2), ang, 1.0)
        img  = cv2.warpAffine(img,  M, (w, h), flags=cv2.INTER_LINEAR,  borderMode=cv2.BORDER_REFLECT)
        mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT)
    if random.random() < 0.5:
        img = img.astype(np.float32)
        img = img * random.uniform(0.85, 1.15) + random.uniform(-0.05, 0.05)*255
        img = np.clip(img, 0, 255).astype(np.uint8)
    return img, mask

class RefugeDS(Dataset):
    def __init__(self, img_dir, msk_dir, train=False, size=IMG_SIZE):
        self.pairs = list_pairs(img_dir, msk_dir); self.train = train; self.size = size
        assert len(self.pairs) > 0, f'No image/mask pairs found in {img_dir} / {msk_dir}'
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        msk = cv2.resize(msk, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        msk = remap_mask(msk)
        if self.train:
            img, msk = augment(img, msk)
        x = img.astype(np.float32)/255.0
        x = (x - IMAGENET_MEAN)/IMAGENET_STD
        x = torch.from_numpy(x.transpose(2,0,1)).float()
        m = torch.from_numpy(msk).long()
        disc_m = (m > 0).long()      # whole disc = rim + cup
        cup_m  = (m == 2).long()     # cup only
        return x, disc_m, cup_m, m

In [ ]:
train_ds = RefugeDS(TRAIN_IMG, TRAIN_MSK, train=True)
val_ds   = RefugeDS(VAL_IMG,   VAL_MSK,   train=False)
test_ds  = RefugeDS(TEST_IMG,  TEST_MSK,  train=False)
print('train/val/test:', len(train_ds), len(val_ds), len(test_ds))
# sanity check on mask encoding
_x,_d,_c,_m = train_ds[0]
print('sample mask unique labels:', torch.unique(_m).tolist(), '| tensor', _x.shape)

train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_ld  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 4 · Model
Blocks: `DoubleConv`, `SpatialAttention` (CBAM spatial), `CrossAttention` (bidirectional, coarsest), `CrossAttentionDown` (last stage, computed on a 48²-pooled residual). The two decoders run **in lockstep** so they can exchange features at the chosen stages.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class SpatialAttention(nn.Module):
    """CBAM spatial attention: gate features by a learned 'where to look' map."""
    def __init__(self, k=7):
        super().__init__(); self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
    def forward(self, x):
        avg = x.mean(1, keepdim=True)
        mx  = x.max(1, keepdim=True)[0]
        a = torch.sigmoid(self.conv(torch.cat([avg, mx], 1)))
        return x * a

class CrossAttention(nn.Module):
    """Bidirectional multi-head cross-attention between disc & cup feature maps."""
    def __init__(self, dim, heads=4):
        super().__init__()
        self.nA = nn.LayerNorm(dim); self.nB = nn.LayerNorm(dim)
        self.a2b = nn.MultiheadAttention(dim, heads, batch_first=True)  # disc queries cup
        self.b2a = nn.MultiheadAttention(dim, heads, batch_first=True)  # cup queries disc
    def forward(self, A, B):
        b, c, h, w = A.shape
        a = A.flatten(2).transpose(1, 2)   # [b, N, c]
        bb = B.flatten(2).transpose(1, 2)
        an, bn = self.nA(a), self.nB(bb)
        a2, _ = self.a2b(an, bn, bn)       # disc attends to cup (pre-update context)
        b2, _ = self.b2a(bn, an, an)       # cup attends to disc
        a = a + a2; bb = bb + b2
        A = a.transpose(1, 2).reshape(b, c, h, w)
        B = bb.transpose(1, 2).reshape(b, c, h, w)
        return A, B

class CrossAttentionDown(nn.Module):
    """Cross-attention at the last (full-res) stage, computed on a pooled grid,
       then the exchange delta is upsampled and added back as a residual."""
    def __init__(self, dim, heads=4, size=48):
        super().__init__(); self.size = size; self.ca = CrossAttention(dim, heads)
    def forward(self, A, B):
        b, c, h, w = A.shape
        Ad = F.adaptive_avg_pool2d(A, self.size)
        Bd = F.adaptive_avg_pool2d(B, self.size)
        Ad2, Bd2 = self.ca(Ad, Bd)
        dA = F.interpolate(Ad2 - Ad, size=(h, w), mode='bilinear', align_corners=False)
        dB = F.interpolate(Bd2 - Bd, size=(h, w), mode='bilinear', align_corners=False)
        return A + dA, B + dB

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.up = nn.ConvTranspose2d(cin, cout, 2, stride=2)
        self.conv = DoubleConv(cout + cskip, cout)
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], 1)
        return self.conv(x)

class Decoder(nn.Module):
    """One decoder pipeline with a spatial-attention block after every stage."""
    def __init__(self):
        super().__init__()
        self.up1 = Up(1024, 512, 512); self.sa1 = SpatialAttention()
        self.up2 = Up(512,  256, 256); self.sa2 = SpatialAttention()
        self.up3 = Up(256,  128, 128); self.sa3 = SpatialAttention()
        self.up4 = Up(128,   64,  64); self.sa4 = SpatialAttention()
        self.head = nn.Conv2d(64, 2, 1)

class DualDecoderSA_CA(nn.Module):
    def __init__(self, heads=4):
        super().__init__()
        # shared encoder
        self.e1 = DoubleConv(3, 64)
        self.e2 = DoubleConv(64, 128)
        self.e3 = DoubleConv(128, 256)
        self.e4 = DoubleConv(256, 512)
        self.bott = DoubleConv(512, 1024)
        self.pool = nn.MaxPool2d(2)
        # two decoders
        self.disc = Decoder()
        self.cup  = Decoder()
        # cross-attention: FIRST stage (48^2, 512ch) and LAST stage (384^2, 64ch via 48^2 pool)
        self.cross_first = CrossAttention(512, heads)
        self.cross_last  = CrossAttentionDown(64, heads, size=48)
    def forward(self, x):
        s1 = self.e1(x)                 # 384, 64
        s2 = self.e2(self.pool(s1))     # 192, 128
        s3 = self.e3(self.pool(s2))     # 96, 256
        s4 = self.e4(self.pool(s3))     # 48, 512
        b  = self.bott(self.pool(s4))   # 24, 1024
        # ---- stage 1 (coarsest, 48^2) ----
        d1 = self.disc.sa1(self.disc.up1(b, s4))
        c1 = self.cup.sa1( self.cup.up1( b, s4))
        d1, c1 = self.cross_first(d1, c1)          # cross-attn @ FIRST stage
        # ---- stage 2 (96^2) ----
        d2 = self.disc.sa2(self.disc.up2(d1, s3))
        c2 = self.cup.sa2( self.cup.up2( c1, s3))
        # ---- stage 3 (192^2) ----
        d3 = self.disc.sa3(self.disc.up3(d2, s2))
        c3 = self.cup.sa3( self.cup.up3( c2, s2))
        # ---- stage 4 (384^2, last) ----
        d4 = self.disc.sa4(self.disc.up4(d3, s1))
        c4 = self.cup.sa4( self.cup.up4( c3, s1))
        d4, c4 = self.cross_last(d4, c4)           # cross-attn @ LAST stage
        return self.disc.head(d4), self.cup.head(c4)

model = DualDecoderSA_CA().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
with torch.no_grad():
    dl, cl = model(torch.zeros(1,3,IMG_SIZE,IMG_SIZE, device=DEVICE))
print('disc logits', tuple(dl.shape), '| cup logits', tuple(cl.shape))
print(f'Trainable params: {n_params:,}')

## 5 · Losses & metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, n=2, eps=1e-6): super().__init__(); self.n=n; self.eps=eps
    def forward(self, logits, target):
        p = F.softmax(logits, 1)
        t = F.one_hot(target, self.n).permute(0,3,1,2).float()
        inter = (p*t).sum((0,2,3)); union = p.sum((0,2,3)) + t.sum((0,2,3))
        dice = (2*inter + self.eps)/(union + self.eps)
        return 1 - dice.mean()

ce = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

def combined(logits, target):     # 0.5 CE + 0.5 Dice
    return 0.5*ce(logits, target) + 0.5*dice_loss(logits, target)

def containment(disc_logits, cup_logits):
    p_disc = F.softmax(disc_logits, 1)[:,1]
    p_cup  = F.softmax(cup_logits, 1)[:,1]
    return F.relu(p_cup - p_disc).mean()

def total_loss(disc_logits, cup_logits, disc_m, cup_m):
    return combined(disc_logits, disc_m) + combined(cup_logits, cup_m) + CONTAIN_W*containment(disc_logits, cup_logits)

@torch.no_grad()
def dice_fg(logits, target):      # foreground (class 1) Dice, hard
    pred = logits.argmax(1)
    p = (pred==1).float(); t = (target==1).float()
    inter = (p*t).sum((1,2)); denom = p.sum((1,2)) + t.sum((1,2))
    d = (2*inter + 1e-6)/(denom + 1e-6)
    return d.mean().item()

## 6 · Training loop (AMP, cosine schedule, early stopping on val mean-FG Dice)

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

hist = {'tr_loss':[], 'val_disc':[], 'val_cup':[], 'val_mean':[]}
best_mean, best_ep, bad = 0.0, -1, 0

for ep in range(1, EPOCHS+1):
    model.train(); running = 0.0
    for x, disc_m, cup_m, _ in train_ld:
        x, disc_m, cup_m = x.to(DEVICE), disc_m.to(DEVICE), cup_m.to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            dl_, cl_ = model(x)
            loss = total_loss(dl_, cl_, disc_m, cup_m)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        running += loss.item()*x.size(0)
    sched.step()
    tr_loss = running/len(train_ds)

    model.eval(); dsum=csum=0.0; nb=0
    with torch.no_grad():
        for x, disc_m, cup_m, _ in val_ld:
            x, disc_m, cup_m = x.to(DEVICE), disc_m.to(DEVICE), cup_m.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
                dl_, cl_ = model(x)
            dsum += dice_fg(dl_, disc_m); csum += dice_fg(cl_, cup_m); nb += 1
    vd, vc = dsum/nb, csum/nb; vm = 0.5*(vd+vc)
    hist['tr_loss'].append(tr_loss); hist['val_disc'].append(vd); hist['val_cup'].append(vc); hist['val_mean'].append(vm)

    tag = ''
    if vm > best_mean:
        best_mean, best_ep, bad = vm, ep, 0
        torch.save({'model':model.state_dict(),'epoch':ep,'val_mean':vm}, CKPT_PATH); tag = '  <- best saved'
    else:
        bad += 1
    print(f'Ep {ep:3d}/{EPOCHS} | loss {tr_loss:.4f} | val disc {vd:.4f} cup {vc:.4f} mean {vm:.4f}{tag}')
    if bad >= PATIENCE:
        print(f'Early stop at epoch {ep} (best mean {best_mean:.4f} @ ep {best_ep}).'); break

print(f'\nBEST val mean-FG Dice = {best_mean:.4f} at epoch {best_ep}')

## 7 · Training curves

In [ ]:
ep = range(1, len(hist['tr_loss'])+1)
fig, ax = plt.subplots(1,2, figsize=(13,4.2))
ax[0].plot(ep, hist['tr_loss']); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, hist['val_disc'], label='disc'); ax[1].plot(ep, hist['val_cup'], label='cup')
ax[1].plot(ep, hist['val_mean'], label='mean(disc,cup)')
ax[1].axhline(0.90, ls='--', c='gray', label='0.90 target')
ax[1].set_title('Validation foreground Dice'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## 8 · Test evaluation + cup-threshold calibration + fused 3-class metrics

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE); model.load_state_dict(ckpt['model']); model.eval()
print('Loaded best checkpoint from epoch', ckpt['epoch'])

@torch.no_grad()
def collect_probs(loader):
    P_disc, P_cup, M = [], [], []
    for x, disc_m, cup_m, m in loader:
        x = x.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            dl_, cl_ = model(x)
        P_disc.append(F.softmax(dl_.float(),1)[:,1].cpu())
        P_cup.append(F.softmax(cl_.float(),1)[:,1].cpu())
        M.append(m)
    return torch.cat(P_disc), torch.cat(P_cup), torch.cat(M)

def dice_from_masks(pred, gt):
    inter = (pred*gt).sum((1,2)); denom = pred.sum((1,2))+gt.sum((1,2))
    return ((2*inter+1e-6)/(denom+1e-6)).mean().item()

# --- calibrate the cup threshold on the VALIDATION set (curbs over-segmentation) ---
vpd, vpc, vm = collect_probs(val_ld)
vcup_gt = (vm==2).float()
best_t, best_cupd = 0.5, 0.0
for t in np.arange(0.30, 0.71, 0.02):
    d = dice_from_masks((vpc>t).float(), vcup_gt)
    if d > best_cupd: best_cupd, best_t = d, float(t)
print(f'Calibrated cup threshold = {best_t:.2f}  (val cup Dice {best_cupd:.4f} vs {dice_from_masks((vpc>0.5).float(), vcup_gt):.4f} at 0.5)')

# --- TEST metrics ---
tpd, tpc, tm = collect_probs(test_ld)
disc_gt = (tm>0).float(); cup_gt = (tm==2).float()
disc_pred = (tpd>0.5).float()
cup_pred_05 = (tpc>0.5).float()
cup_pred_cal = (tpc>best_t).float()

wd = dice_from_masks(disc_pred, disc_gt)
cc05 = dice_from_masks(cup_pred_05, cup_gt)
cccal = dice_from_masks(cup_pred_cal, cup_gt)
print(f'TEST  whole-disc Dice : {wd:.4f}')
print(f'TEST  cup Dice @0.5   : {cc05:.4f}   | mean(disc,cup) {0.5*(wd+cc05):.4f}')
print(f'TEST  cup Dice @cal   : {cccal:.4f}   | mean(disc,cup) {0.5*(wd+cccal):.4f}')

# --- fused 3-class (cup must lie inside disc) using calibrated cup ---
cup_in = (cup_pred_cal.bool() & disc_pred.bool()).float()
rim_gt = (tm==1).float(); rim_pred = (disc_pred.bool() & ~cup_in.bool()).float()
print(f'TEST  fused rim Dice  : {dice_from_masks(rim_pred, rim_gt):.4f}')
print(f'TEST  fused cup Dice  : {dice_from_masks(cup_in, cup_gt):.4f}')

## 9 · Visual outputs (Input | GT | Disc head | Cup head | Fused)

In [ ]:
import matplotlib.patches as mpatches
inv_mean = IMAGENET_MEAN; inv_std = IMAGENET_STD
def denorm(x):
    img = x.numpy().transpose(1,2,0)*inv_std + inv_mean
    return np.clip(img,0,1)

n_show = 4
xb, db, cb, mb = next(iter(test_ld))
with torch.no_grad():
    dl_, cl_ = model(xb.to(DEVICE))
    pdisc = (F.softmax(dl_.float(),1)[:,1].cpu()>0.5)
    pcup  = (F.softmax(cl_.float(),1)[:,1].cpu()>best_t)
fig, ax = plt.subplots(n_show, 5, figsize=(13, 2.7*n_show))
cmap3 = np.array([[64,0,90],[26,133,133],[240,228,66]])/255.0  # bg, rim, cup
for i in range(n_show):
    gt = mb[i].numpy()
    fused = np.zeros_like(gt); fused[pdisc[i].numpy()]=1; fused[(pcup[i].numpy() & pdisc[i].numpy())]=2
    ax[i,0].imshow(denorm(xb[i]));            ax[i,0].set_title('Input' if i==0 else '')
    ax[i,1].imshow(cmap3[gt]);                ax[i,1].set_title('GT' if i==0 else '')
    ax[i,2].imshow(pdisc[i], cmap='gray');    ax[i,2].set_title('Disc head' if i==0 else '')
    ax[i,3].imshow(pcup[i], cmap='gray');     ax[i,3].set_title('Cup head' if i==0 else '')
    ax[i,4].imshow(cmap3[fused]);             ax[i,4].set_title('Fused' if i==0 else '')
    for j in range(5): ax[i,j].axis('off')
plt.tight_layout(); plt.show()

## Notes
- **If you hit CUDA OOM at 384:** set `BATCH = 2` in the CONFIG cell (AMP is already on). You can also reduce cross-attention cost by lowering `size=48` to `32` in `CrossAttentionDown` and in the first-stage attention if needed.
- **Loss:** per-head `0.5·CE + 0.5·Dice` plus a containment term (`cup ⊆ disc`). The test cell also does a **validation-based cup-threshold calibration**, which is a cheap, no-retrain lift on cup Dice — exactly the over-segmentation fix discussed.
- **Selection:** best checkpoint on validation `mean(disc,cup)` foreground Dice, early stop patience 25.
- This mirrors your week-7 protocol (official REFUGE 400/400/400) but at 384 and with **both** attention mechanisms combined.
